# SmolLM2 AM-CeNN Hybrid v3 — Colab

Version 3 keeps **exact local causal attention** for recent tokens and uses a **gated recurrent AM-CeNN memory** only for older context. During layerwise calibration Q/K/V/O stay frozen; during global distillation they use a much smaller learning rate than the new AM-CeNN parameters. The dense pretrained FFN is left unchanged.

This notebook trains, evaluates against the original SmolLM2-135M teacher, prints side-by-side generations, and uses the repository's mandatory private Hugging Face backup.

In [ ]:
import pathlib, subprocess, sys

REPO_DIR = pathlib.Path("/content/TinyCeNN-LM")
if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "reset", "--hard", "origin/main"], check=True)
else:
    subprocess.run(["git", "clone", "https://github.com/vtavakkoli/TinyCeNN-LM.git", str(REPO_DIR)], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_DIR)], check=True)

# Import in the notebook kernel so the parent live/HF backup wrapper is active.
import tinycenn_lm

print("Repository ready:", REPO_DIR)

In [ ]:
import os
from google.colab import userdata
from huggingface_hub import login

token = userdata.get("HF_TOKEN")
if not token:
    raise RuntimeError("Add a Hugging Face WRITE token to Colab Secrets as HF_TOKEN, then rerun this cell.")

os.environ["HF_TOKEN"] = token
login(token=token, add_to_git_credential=False)
print("Hugging Face login ready; mandatory backup is enabled.")

In [ ]:
from datetime import datetime, timezone

BASE_MODEL = "HuggingFaceTB/SmolLM2-135M"
CONTEXT_LENGTH = 128
LOCAL_WINDOW = 32
FEATURE_DIM = 256
GROUP_SIZE = 1
CALIBRATION_STEPS = 20
FINAL_MAX_TOKENS = 500_000
MAX_RUNTIME_MINUTES = 45

run_stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
OUTPUT_DIR = REPO_DIR / "checkpoints" / f"smollm2-amcenn-hybrid-v3-{run_stamp}"
EVAL_JSON = OUTPUT_DIR / "v3_evaluation.json"

print("Output:", OUTPUT_DIR)

## Train v3

The defaults below are a practical Colab test. For a stronger run after the first validation, increase `FINAL_MAX_TOKENS` to 1–2M and `CALIBRATION_STEPS` to 30–50.

In [ ]:
cmd = [
    sys.executable, str(REPO_DIR / "scripts" / "train_smollm2_amcenn_v3.py"),
    "--base-model", BASE_MODEL,
    "--output-dir", str(OUTPUT_DIR),
    "--context-length", str(CONTEXT_LENGTH),
    "--local-window", str(LOCAL_WINDOW),
    "--feature-dim", str(FEATURE_DIM),
    "--group-size", str(GROUP_SIZE),
    "--calibration-steps", str(CALIBRATION_STEPS),
    "--calibration-lr", "1e-4",
    "--global-gate-init", "0.05",
    "--global-gate-cap", "0.35",
    "--final-max-tokens", str(FINAL_MAX_TOKENS),
    "--final-lr", "5e-5",
    "--qkvo-lr", "3e-6",
    "--final-eval-batches", "8",
    "--max-runtime-minutes", str(MAX_RUNTIME_MINUTES),
]
print(" ".join(cmd))
subprocess.run(cmd, check=True)

## Compare v3 directly with the original teacher

This evaluates held-out CE/perplexity/KL and prints the same four prompts for teacher and student.

In [ ]:
eval_cmd = [
    sys.executable, str(REPO_DIR / "scripts" / "evaluate_smollm2_amcenn_v3.py"),
    "--model-dir", str(OUTPUT_DIR),
    "--output", str(EVAL_JSON),
    "--eval-tokens", "8192",
    "--context-length", str(CONTEXT_LENGTH),
    "--max-new-tokens", "64",
]
print(" ".join(eval_cmd))
subprocess.run(eval_cmd, check=True)

In [ ]:
import json
import pandas as pd
from IPython.display import display

report = json.loads(EVAL_JSON.read_text())
summary = pd.DataFrame([{
    "teacher CE": report["teacher_ce"],
    "student CE": report["student_ce"],
    "CE gap": report["ce_gap"],
    "teacher PPL": report["teacher_perplexity"],
    "student PPL": report["student_perplexity"],
    "student↔teacher KL": report["student_teacher_kl"],
    "mean global gate": report["gate_stats"]["mean_global_gate"],
}])
display(summary)

for item in report["generations"]:
    print("=" * 100)
    print("PROMPT:", item["prompt"])
    print("\nTEACHER:\n", item["teacher"])
    print("\nV3 STUDENT:\n", item["student"])
    print(
        f"\n3-gram repetition teacher={item['teacher_repeat_3gram_fraction']:.3f} "
        f"student={item['student_repeat_3gram_fraction']:.3f}"
    )